# L13 · HTTP 与 REST：互联网的对话协议

**学习目标**
- 理解 HTTP 是什么：客户端和服务器的「问答协议」
- 理解「请求/响应」「方法 GET/POST」「URL」「JSON」
- 亲手发起一次真实 HTTP 请求并解析响应

**前置依赖**：L01-L06（函数、循环、字典）  
**预计时长**：40 分钟  
**技术栈**：Python 标准库 `http.server`、`urllib`（零第三方依赖）

---

## 概念讲解：互联网 = 全世界在互相「点餐」

你打开 App，背后是手机（**客户端**）给服务器发了一条「我要这个数据」的消息（**请求**），
服务器算好后再回一条消息（**响应**）。这条消息的格式规矩，就叫 **HTTP**。

- **URL**：你要的「菜」的地址，如 `/users/1`
- **方法**：你想对这道菜做什么 —— `GET`(取)、`POST`(新建)、`PUT`(改)、`DELETE`(删)
- **JSON**：双方都看得懂的「数据便签」（长得像 Python 字典）

这套「用 URL + 方法操作资源」的风格，叫 **REST**，是现代 API 的通用语言。

## 第一步：在笔记本里起一个迷你服务器

In [ ]:
import http.server, socketserver, threading, json, time

# 我们造一个「回声服务器」：你发什么 JSON，它还你什么，并加个时间戳
class Handler(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        body = json.loads(self.rfile.read(length) or b"{}")
        reply = {"你发来的": body, "服务器时间": time.strftime("%H:%M:%S")}
        data = json.dumps(reply, ensure_ascii=False).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(data)
    def log_message(self, *a): pass   # 安静模式

PORT = 8765
httpd = socketserver.TCPServer(("127.0.0.1", PORT), Handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"✅ 迷你服务器已在 http://127.0.0.1:{PORT} 启动（后台线程）")

## 第二步：当客户端，发起一次 HTTP 请求

In [ ]:
import urllib.request

req = urllib.request.Request(
    f"http://127.0.0.1:{PORT}",
    data=json.dumps({"name": "小明", "msg": "你好服务器"}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req) as resp:
    text = resp.read().decode("utf-8")
print("🔁 服务器回复：")
print(json.dumps(json.loads(text), ensure_ascii=False, indent=2))

# 🎯 AHA 顿悟单元格：你刚刚完成了一次「跨进程互联网对话」

运行本单元格前，请先运行上面的「第一步」和「第二步」。
你会看到：**你笔记本里的代码（客户端）和另一个后台进程（服务器）真的通过 HTTP 聊上了天**。
改 `body` 里的内容再发一次，服务器会原样回声并附上它的时间。

> 你刚才亲手实现了「前端↔后端」最底层的那一下握手。抖音、微信、ChatGPT 的每一次交互，
> 本质上都是这样的「请求→响应」。你已经站在了全栈的大门里。

In [ ]:
# ===== 运行我！（确认上面服务器已启动）=====
import urllib.request, json
body = {"action": "打招呼", "from": "学员", "feeling": "兴奋"}
req = urllib.request.Request(
    f"http://127.0.0.1:{PORT}",
    data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
    headers={"Content-Type": "application/json"}, method="POST")
with urllib.request.urlopen(req) as resp:
    reply = json.loads(resp.read().decode("utf-8"))
print("  📡 请求已发出，服务器回应：")
print("  " + json.dumps(reply, ensure_ascii=False, indent=2).replace("\n", "\n  "))
print("  ✨ 恭喜：你完成了人生第一次 HTTP 跨进程对话！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：客户端/服务器分离心智；HTTP 异步感（先起服务再请求）。  
**易错点**：服务器须先运行，否则 urllib 报连接拒绝；端口冲突（用 8765 高位端口）。  
**AHA 机制**：同 notebook 内起服务+请求，零依赖证明「跨进程对话」真实发生，强全栈破冰感。  
**衔接**：L14 FastAPI（把本课的玩具服务器换成工业级框架）；L15 请求响应细节。  
**注意**：`socketserver.TCPServer` 单线程但 daemon 线程足够演示；生产用 ThreadingTCPServer。  
**收尾**：本笔记本末尾 `httpd.shutdown()` 可选，daemon 线程随 kernel 退出。

# 📚 作业 / 下一步

1. 改 `body` 里的内容，多发几次，观察服务器回声。
2. 思考：如果服务器「记住」你上次发的内容（存到一个列表里），它就变成了有状态的 API——这正是 L16 数据库要做的。
3. 下一课 **L14 FastAPI 入门：写出你的第一个 API** —— 用工业级框架，三行代码上线一个真服务。